# Derived Tables: Simple OHCO

This notebook follows the M05 class workflow for creating `BOW`, `DTM`, `TFIDF`, and reduced L2-normalized `TFIDF_L2` tables from the simple-OHCO parsed tables.

# Set Up

## Import

In [1]:
import pandas as pd
import numpy as np

## Config

In [2]:
data_home = "."
output_dir = "output"
data_prefix = "constitutions-simple-ohco"
source_dir = output_dir
CSV_DELIM = "|"

In [3]:
OHCO = [
    "constitution_id",
    "provision_num",
    "para_num",
    "sent_num",
    "token_num",
]

bags = dict(
    SENTS=OHCO[:4],
    PARAS=OHCO[:3],
    PROVISIONS=OHCO[:2],
    CONSTITUTIONS=OHCO[:1],
)

In [4]:
bag = "CONSTITUTIONS"
# bag = "PROVISIONS"
# bag = "PARAS"
# bag = "SENTS"

# Prepare the Data

## Import Tables

This matches the M05 pattern of reading the parsed M04-style output tables. This notebook uses the simple-OHCO tables created by `Parsed_and_Annotated_Data_Simple_OHCO.ipynb`.

In [5]:
LIB = pd.read_csv(f"{source_dir}/{data_prefix}-LIB.csv", sep=CSV_DELIM).set_index("constitution_id")
TOKEN = pd.read_csv(f"{source_dir}/{data_prefix}-CORPUS.csv", sep=CSV_DELIM).set_index(OHCO)
VOCAB = pd.read_csv(f"{source_dir}/{data_prefix}-VOCAB.csv", sep=CSV_DELIM).set_index("term_str")

In [6]:
TOKEN = TOKEN.dropna(subset=["term_str"]).copy()
TOKEN = TOKEN.loc[TOKEN.term_str.astype(str).str.len() > 0]
TOKEN.head()

token_str  \
constitution_id  provision_num para_num sent_num token_num             
Afghanistan_2004 1             1        1        1          Preamble   
                               2        1        1                In   
                                                 2               the   
                                                 3              name   
                                                 4                of   

                                                            term_str pos  \
constitution_id  provision_num para_num sent_num token_num                 
Afghanistan_2004 1             1        1        1          preamble  JJ   
                               2        1        1                in  IN   
                                                 2               the  DT   
                                                 3              name  NN   
                                                 4                of  IN   

                                                           pos_group  \
constitution_id  provision_num para_num sent_num token_num             
Afghanistan_2004 1             1        1        1                JJ   
                               2        1        1                IN   
                                                 2                DT   
                                                 3                NN   
                                                 4                IN   

                                                           pos_group_name  
constitution_id  provision_num para_num sent_num token_num                 
Afghanistan_2004 1             1        1        1                    ADJ  
                               2        1        1                   PREP  
                                                 2                    DET  
                                                 3                   NOUN  
                                                 4                   PREP

In [7]:
VOCAB = VOCAB.sort_values("n", ascending=False).copy()
VOCAB["term_rank"] = np.arange(1, VOCAB.shape[0] + 1)
VOCAB["p"] = VOCAB.n / VOCAB.n.sum()
VOCAB["i"] = -np.log2(VOCAB.p)
VOCAB.head()

,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,cat_pos_group,n_pos,cat_pos,stop,porter_stem,df,idf,dfidf,dp,di,dh,term_rank
term_str,,,,,,,,,,,,,,,,,,,
the,432334,3,0.100629,3.312877,DT,DT,3,DT NN VB,3,DT NNP VBP,1,the,192,0.0,0.0,1.0,0.0,0.0,1
of,298538,2,0.069487,3.847108,IN,IN,6,JJ IN NN VB CC CD,7,JJ IN NN NNP CC VBP CD,1,of,192,0.0,0.0,1.0,0.0,0.0,2
and,130658,3,0.030412,5.039227,CC,CC,4,IN NN VB CC,4,IN NNP CC VBP,1,and,192,0.0,0.0,1.0,0.0,0.0,3
to,118070,2,0.027482,5.185380,TO,TO,4,NN VB TO CD,4,NNP TO VBP CD,1,to,192,0.0,0.0,1.0,0.0,0.0,4
in,89867,2,0.020917,5.579159,IN,IN,2,IN NN,2,IN NNP,1,in,192,0.0,0.0,1.0,0.0,0.0,5


# Create BOW from TOKEN

Following M05, create `BOW` with a split-apply-combine pattern by grouping on the selected bag plus `term_str`.

In [8]:
bags[bag] + ["term_str"]

['constitution_id', 'term_str']

In [9]:
BOW = TOKEN.groupby(bags[bag] + ["term_str"]).term_str.count().to_frame("n")
BOW.head()

n
constitution_id  term_str    
Afghanistan_2004 1         21
                 10         4
                 100        1
                 101        1
                 102        1

# Document-Term Count Matrix

M05 calls this `DTCM`. The final project calls it `DTM`, so this notebook keeps both names: `DTCM` for class consistency and `DTM` for the project table.

In [10]:
DTCM = BOW.n.unstack(fill_value=0)
DTM = DTCM.astype(pd.SparseDtype("int", 0))
DTCM.head(10)

term_str,0,00,000,00000,001,002,003,005,006,01,...,zr,zs,zt,zuba,zug,zui,zulia,zurich,zurmi,zuru
constitution_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Albania_2008,0,0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Algeria_2008,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Andorra_1993,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Angola_2010,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Antigua_and_Barbuda_1981,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Argentina_1994,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Armenia_2005,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Australia_1985,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Types and Tokens by DOC

In [11]:
DOC = DTCM.sum(1).to_frame("n_tokens")
DOC["n_types"] = DTCM.astype("bool").sum(1)
DOC["pkr"] = DOC.n_types / DOC.n_tokens

if bag == "CONSTITUTIONS":
    DOC = DOC.join(LIB[["country", "file_year", "original_year", "revision_year", "title"]])

DOC.sort_values("pkr").head(20)

/var/folders/2n/rqnjq39x0bl3l55y0ghnzcz00000gn/T/ipykernel_71607/1205689900.py:1: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  DOC = DTCM.sum(1).to_frame("n_tokens")
/var/folders/2n/rqnjq39x0bl3l55y0ghnzcz00000gn/T/ipykernel_71607/1205689900.py:2: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  DOC["n_types"] = DTCM.astype("bool").sum(1)


,n_tokens,n_types,pkr,country,file_year,original_year,revision_year,title
constitution_id,,,,,,,,
India_2012,103941,3919,0.037704,India,2012,1949.0,2012.0,India
St_Kitts_and_Nevis_1983,46896,2106,0.044908,St Kitts and Nevis,1983,1983.0,NaN,St. Kitts and Nevis
Malaysia_1996,64746,3123,0.048235,Malaysia,1996,1957.0,1996.0,Malaysia
St_Lucia_1978,40394,1998,0.049463,St Lucia,1978,1978.0,NaN,St. Lucia
Sweden_2012,59921,3066,0.051167,Sweden,2012,1974.0,2012.0,Sweden
Jamaica_1994,38422,1974,0.051377,Jamaica,1994,1962.0,1994.0,Jamaica
Fiji_2013,47371,2448,0.051677,Fiji,2013,2013.0,NaN,Fiji
Singapore_2010,44342,2307,0.052027,Singapore,2010,1959.0,2010.0,Singapore
Lesotho_1998,42910,2266,0.052808,Lesotho,1998,1993.0,1998.0,Lesotho


# Compute TFIDF

In [12]:
tf_method = "double_norm"       # sum, max, log, double_norm, raw, binary
tf_norm_k = .5                  # only used for double_norm
idf_method = "standard"        # standard, max, plus, smooth

## Compute TF

In [13]:
print("TF method:", tf_method)

if tf_method == "sum":
    TF = DTCM.T / DTCM.T.sum()

elif tf_method == "smooth":
    TF = (DTCM.T / DTCM.T.sum()) + 1

elif tf_method == "max":
    TF = DTCM.T / DTCM.T.max()

elif tf_method == "log":
    TF = np.log2(1 + DTCM.T)

elif tf_method == "raw":
    TF = DTCM.T

elif tf_method == "double_norm":
    TF = ((DTCM.T + tf_norm_k) / (DTCM.T.max() + tf_norm_k)) + tf_norm_k

elif tf_method == "binary":
    TF = DTCM.T.astype("bool").astype("int")

TF = TF.T
TF.head()

TF method: double_norm


term_str,0,00,000,00000,001,002,003,005,006,01,...,zr,zs,zt,zuba,zug,zui,zulia,zurich,zurmi,zuru
constitution_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0.500439,0.500439,0.500439,0.500439,0.500439,0.500439,0.500439,0.500439,0.500439,0.500439,...,0.500439,0.500439,0.500439,0.500439,0.500439,0.500439,0.500439,0.500439,0.500439,0.500439
Albania_2008,0.500321,0.500321,0.501603,0.500321,0.500321,0.500321,0.500321,0.500321,0.500321,0.500321,...,0.500321,0.500321,0.500321,0.500321,0.500321,0.500321,0.500321,0.500321,0.500321,0.500321
Algeria_2008,0.500318,0.500318,0.500318,0.500318,0.500318,0.500318,0.500318,0.500318,0.500318,0.500318,...,0.500318,0.500318,0.500318,0.500318,0.500318,0.500318,0.500318,0.500318,0.500318,0.500318
Andorra_1993,0.500447,0.500447,0.500447,0.500447,0.500447,0.500447,0.500447,0.500447,0.500447,0.500447,...,0.500447,0.500447,0.500447,0.500447,0.500447,0.500447,0.500447,0.500447,0.500447,0.500447
Angola_2010,0.500175,0.500175,0.500525,0.500175,0.500175,0.500175,0.500175,0.500175,0.500175,0.500175,...,0.500175,0.500175,0.500175,0.500175,0.500175,0.500175,0.500175,0.500175,0.500175,0.500175


## Compute DF

In [14]:
DF = DTCM.astype("bool").sum()
DF.sort_values(ascending=False).head(20)

term_str
by        192
the       192
6         192
who       192
is        192
which     192
it        192
its       192
his       192
order     192
for       192
10        192
1         192
rights    192
15        192
as        192
are       192
no        192
of        192
their     192
dtype: int64

## Compute IDF

In [15]:
N = DTCM.shape[0]

In [16]:
print("IDF method:", idf_method)

if idf_method == "standard":
    IDF = np.log2(N / DF)

elif idf_method == "max":
    IDF = np.log2(DF.max() / DF)

elif idf_method == "plus":
    IDF = np.log2(N / DF) + 1

elif idf_method == "smooth":
    IDF = np.log2((1 + N) / (1 + DF)) + 1

IDF.sort_values(ascending=False).head(20)

IDF method: standard


term_str
istahili       7.584963
loaded         7.584963
litige         7.584963
litis          7.584963
littoral       7.584963
litunga        7.584963
liv            7.584963
livelihoods    7.584963
livingston     7.584963
ljubljana      7.584963
ljupco         7.584963
llenar         7.584963
llevar         7.584963
ln             7.584963
load           7.584963
lobby          7.584963
litigating     7.584963
lobbying       7.584963
lobo           7.584963
locais         7.584963
dtype: float64

## Compute TFIDF

In [17]:
TFIDF = TF * IDF
TFIDF.head()

term_str,0,00,000,00000,001,002,003,005,006,01,...,zr,zs,zt,zuba,zug,zui,zulia,zurich,zurmi,zuru
constitution_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,2.209456,2.794932,1.016078,3.795809,3.295371,3.295371,3.795809,3.795809,3.795809,2.209456,...,3.795809,3.795809,3.795809,3.795809,3.795809,3.795809,3.295371,3.295371,3.795809,3.795809
Albania_2008,2.208934,2.794272,1.018442,3.794913,3.294592,3.294592,3.794913,3.794913,3.794913,2.208934,...,3.794913,3.794913,3.794913,3.794913,3.794913,3.794913,3.294592,3.294592,3.794913,3.794913
Algeria_2008,2.208921,2.794255,1.015832,3.794890,3.294572,3.294572,3.794890,3.794890,3.794890,2.208921,...,3.794890,3.794890,3.794890,3.794890,3.794890,3.794890,3.294572,3.294572,3.794890,3.794890
Andorra_1993,2.209491,2.794976,1.016094,3.795869,3.295422,3.295422,3.795869,3.795869,3.795869,2.209491,...,3.795869,3.795869,3.795869,3.795869,3.795869,3.795869,3.295422,3.295422,3.795869,3.795869
Angola_2010,2.208292,2.793459,1.016253,3.793809,3.293634,3.293634,3.793809,3.793809,3.793809,2.208292,...,3.793809,3.793809,3.793809,3.793809,3.793809,3.793809,3.293634,3.293634,3.793809,3.793809


## Move Things to Their Places

In [18]:
VOCAB["df"] = DF
VOCAB["idf"] = IDF
VOCAB["tfidf_mean"] = TFIDF.mean()

BOW["tf"] = TF.stack().reindex(BOW.index)
BOW["tfidf"] = TFIDF.stack().reindex(BOW.index)

BOW.head()

n        tf     tfidf
constitution_id  term_str                        
Afghanistan_2004 1         21  0.518868  0.000000
                 10         4  0.503949  0.000000
                 100        1  0.501316  0.114005
                 101        1  0.501316  0.122879
                 102        1  0.501316  0.127357

# Apply

In [19]:
TFIDF_BY_CONSTITUTION = TFIDF.groupby("constitution_id").mean()

def top_constitutions_for_term(term_str):
    return TFIDF_BY_CONSTITUTION[term_str]        .to_frame("tfidf_mean")        .join(LIB)        .sort_values("tfidf_mean", ascending=False)        [["country", "file_year", "title", "tfidf_mean"]]

top_constitutions_for_term("rights").head(20)

,country,file_year,title,tfidf_mean
constitution_id,,,,
Afghanistan_2004,Afghanistan,2004,Afghanistan,0.0
Albania_2008,Albania,2008,Albania,0.0
Nepal_2010,Nepal,2010,Nepal,0.0
Netherlands_2008,Netherlands,2008,Netherlands,0.0
Nicaragua_2005,Nicaragua,2005,Nicaragua,0.0
Niger_2010,Niger,2010,Niger,0.0
Nigeria_1999,Nigeria,1999,Nigeria,0.0
Norway_2004,Norway,2004,Norway,0.0
Oman_2011,Oman,2011,Oman,0.0


# Apply Aggregates to VOCAB

This follows the M05 aggregate TFIDF notebook. `dfidf` depends only on document frequency, so ties are expected.

In [20]:
VOCAB["dfidf"] = VOCAB.df * VOCAB.idf
VOCAB["dp"] = VOCAB.df / N
VOCAB["di"] = np.log2(1 / VOCAB.dp)
VOCAB["dh"] = VOCAB.dp * VOCAB.di

VOCAB.sort_values(["dfidf", "n"], ascending=False).head(20)

,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,cat_pos_group,n_pos,cat_pos,stop,porter_stem,df,idf,dfidf,dp,di,dh,term_rank,tfidf_mean
term_str,,,,,,,,,,,,,,,,,,,,
assent,623,6,0.000145,12.751575,NN,NN,3,JJ NN VB,5,JJ NN NNP VB VBD,0,assent,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,742,0.719796
labor,500,5,0.000116,13.068879,NN,NN,1,NN,2,NN NNP,0,labor,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,884,0.719653
rural,402,5,0.000094,13.383612,JJ,JJ,3,JJ NN VB,3,JJ NNP VB,0,rural,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,1047,0.718994
hand,348,4,0.000081,13.591720,NN,NN,1,NN,2,NN NNP,0,hand,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,1182,0.719097
servants,347,8,0.000081,13.595871,NNS,NN,2,NN VB,5,NN NNP NNS VBZ NNPS,0,servant,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,1187,0.719190
like,303,4,0.000071,13.791489,JJ,JJ,3,JJ IN VB,3,JJ IN VB,0,like,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,1320,0.718882
organized,299,9,0.000070,13.810662,VBN,VB,3,JJ NN VB,4,JJ NNP VBN VBD,0,organ,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,1330,0.719061
class,291,5,0.000068,13.849788,NN,NN,1,NN,2,NN NNP,0,class,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,1363,0.718948
fees,251,4,0.000058,14.063120,NNS,NN,1,NN,1,NNS,0,fee,71.0,1.435215,101.900292,0.369792,1.435215,0.530731,1547,0.718750


# Reduce VOCAB

Following M05, use document entropy `dh` to define a threshold for significant terms.

In [21]:
thresh = VOCAB.dh.quantile(.9).round(3)
float(thresh)

0.437

In [22]:
SIGS = VOCAB[VOCAB.dh >= thresh].sort_values("dh", ascending=False)
SIGS.shape

(2410, 20)

In [23]:
SIGS[["max_pos", "max_pos_group", "n", "i", "df", "idf", "dfidf", "dh"]].head(25)

,max_pos,max_pos_group,n,i,df,idf,dfidf,dh
term_str,,,,,,,,
honor,NN,NN,179,14.550848,71.0,1.435215,101.900292,0.530731
labor,NN,NN,500,13.068879,71.0,1.435215,101.900292,0.530731
occasion,NN,NN,221,14.246761,71.0,1.435215,101.900292,0.530731
stage,NN,NN,106,15.306743,71.0,1.435215,101.900292,0.530731
integral,JJ,JJ,180,14.542810,71.0,1.435215,101.900292,0.530731
construction,NN,NN,207,14.341176,71.0,1.435215,101.900292,0.530731
class,NN,NN,291,13.849788,71.0,1.435215,101.900292,0.530731
rural,JJ,JJ,402,13.383612,71.0,1.435215,101.900292,0.530731
reading,NN,NN,220,14.253304,71.0,1.435215,101.900292,0.530731


# Reduced and Normalized TFIDF_L2

In [24]:
TFIDF_REDUCED = TFIDF[SIGS.index]
TFIDF_REDUCED.head()

term_str,honor,labor,occasion,stage,integral,construction,class,rural,reading,delivered,...,engage,punished,functioning,includes,affect,maximum,f,equally,restrictions,needs
constitution_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0.718237,0.720756,0.718237,0.718237,0.718237,0.718237,0.718237,0.718237,0.718237,0.718237,...,0.365086,0.365724,0.363810,0.364448,0.363810,0.364448,0.363810,0.363810,0.364448,0.363810
Albania_2008,0.719908,0.722669,0.718068,0.718068,0.718068,0.718068,0.718068,0.718068,0.718068,0.718068,...,0.363724,0.364190,0.363724,0.364190,0.364190,0.363724,0.368852,0.364190,0.363724,0.364656
Algeria_2008,0.719887,0.718063,0.718975,0.718063,0.718975,0.718975,0.718063,0.718063,0.718063,0.718975,...,0.363722,0.365107,0.368339,0.363722,0.363722,0.366492,0.363722,0.363722,0.363722,0.363722
Andorra_1993,0.718249,0.718249,0.718249,0.718249,0.718249,0.718249,0.718249,0.718249,0.718249,0.718249,...,0.363815,0.363815,0.370309,0.363815,0.363815,0.365764,0.367712,0.363815,0.363815,0.364465
Angola_2010,0.717859,0.717859,0.718362,0.717859,0.718362,0.718362,0.717859,0.719869,0.717859,0.718362,...,0.364127,0.363873,0.373038,0.363873,0.364127,0.363873,0.372020,0.363873,0.365655,0.364382


In [25]:
row_norms = np.sqrt((TFIDF_REDUCED ** 2).sum(1))
TFIDF_L2 = TFIDF_REDUCED.div(row_norms.replace(0, np.nan), axis=0).fillna(0)
TFIDF_L2.head()

/var/folders/2n/rqnjq39x0bl3l55y0ghnzcz00000gn/T/ipykernel_71607/3970364029.py:1: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  row_norms = np.sqrt((TFIDF_REDUCED ** 2).sum(1))


term_str,honor,labor,occasion,stage,integral,construction,class,rural,reading,delivered,...,engage,punished,functioning,includes,affect,maximum,f,equally,restrictions,needs
constitution_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0.016776,0.016835,0.016776,0.016776,0.016776,0.016776,0.016776,0.016776,0.016776,0.016776,...,0.008528,0.008542,0.008498,0.008513,0.008498,0.008513,0.008498,0.008498,0.008513,0.008498
Albania_2008,0.016820,0.016885,0.016777,0.016777,0.016777,0.016777,0.016777,0.016777,0.016777,0.016777,...,0.008498,0.008509,0.008498,0.008509,0.008509,0.008498,0.008618,0.008509,0.008498,0.008520
Algeria_2008,0.016821,0.016778,0.016800,0.016778,0.016800,0.016800,0.016778,0.016778,0.016778,0.016800,...,0.008499,0.008531,0.008607,0.008499,0.008499,0.008564,0.008499,0.008499,0.008499,0.008499
Andorra_1993,0.016777,0.016777,0.016777,0.016777,0.016777,0.016777,0.016777,0.016777,0.016777,0.016777,...,0.008498,0.008498,0.008650,0.008498,0.008498,0.008544,0.008589,0.008498,0.008498,0.008513
Angola_2010,0.016773,0.016773,0.016785,0.016773,0.016785,0.016785,0.016773,0.016820,0.016773,0.016785,...,0.008508,0.008502,0.008716,0.008502,0.008508,0.008502,0.008692,0.008502,0.008544,0.008514


In [26]:
np.sqrt((TFIDF_L2 ** 2).sum(1)).describe()

/var/folders/2n/rqnjq39x0bl3l55y0ghnzcz00000gn/T/ipykernel_71607/911463411.py:1: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  np.sqrt((TFIDF_L2 ** 2).sum(1)).describe()


count    1.920000e+02
mean     1.000000e+00
std      3.859777e-15
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

# Inspect Tables

In [27]:
BOW.head(20)

n        tf     tfidf
constitution_id  term_str                        
Afghanistan_2004 1         21  0.518868  0.000000
                 10         4  0.503949  0.000000
                 100        1  0.501316  0.114005
                 101        1  0.501316  0.122879
                 102        1  0.501316  0.127357
                 103        1  0.501316  0.140961
                 104        1  0.501316  0.145553
                 105        1  0.501316  0.150174
                 106        1  0.501316  0.154826
                 107        1  0.501316  0.159507
                 108        1  0.501316  0.164219
                 109        1  0.501316  0.164219
                 11         2  0.502194  0.003783
                 110        1  0.501316  0.173735
                 111        2  0.502194  0.169257
                 112        2  0.502194  0.178853
                 113        1  0.501316  0.188249
                 114        1  0.501316  0.193152
                 115        1  0.501316  0.198089
                 116        1  0.501316  0.208065

In [28]:
DTM.head(10)

term_str,0,00,000,00000,001,002,003,005,006,01,...,zr,zs,zt,zuba,zug,zui,zulia,zurich,zurmi,zuru
constitution_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Albania_2008,0,0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Algeria_2008,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Andorra_1993,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Angola_2010,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Antigua_and_Barbuda_1981,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Argentina_1994,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Armenia_2005,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Australia_1985,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [29]:
TFIDF.head(10)

term_str,0,00,000,00000,001,002,003,005,006,01,...,zr,zs,zt,zuba,zug,zui,zulia,zurich,zurmi,zuru
constitution_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,2.209456,2.794932,1.016078,3.795809,3.295371,3.295371,3.795809,3.795809,3.795809,2.209456,...,3.795809,3.795809,3.795809,3.795809,3.795809,3.795809,3.295371,3.295371,3.795809,3.795809
Albania_2008,2.208934,2.794272,1.018442,3.794913,3.294592,3.294592,3.794913,3.794913,3.794913,2.208934,...,3.794913,3.794913,3.794913,3.794913,3.794913,3.794913,3.294592,3.294592,3.794913,3.794913
Algeria_2008,2.208921,2.794255,1.015832,3.794890,3.294572,3.294572,3.794890,3.794890,3.794890,2.208921,...,3.794890,3.794890,3.794890,3.794890,3.794890,3.794890,3.294572,3.294572,3.794890,3.794890
Andorra_1993,2.209491,2.794976,1.016094,3.795869,3.295422,3.295422,3.795869,3.795869,3.795869,2.209491,...,3.795869,3.795869,3.795869,3.795869,3.795869,3.795869,3.295422,3.295422,3.795869,3.795869
Angola_2010,2.208292,2.793459,1.016253,3.793809,3.293634,3.293634,3.793809,3.793809,3.793809,2.208292,...,3.793809,3.793809,3.793809,3.793809,3.793809,3.793809,3.293634,3.293634,3.793809,3.793809
Antigua_and_Barbuda_1981,2.208252,2.793409,1.015524,3.793741,3.293575,3.293575,3.793741,3.793741,3.793741,2.208252,...,3.793741,3.793741,3.793741,3.793741,3.793741,3.793741,3.293575,3.293575,3.793741,3.793741
Argentina_1994,2.209093,2.794472,1.017358,3.795185,3.294829,3.294829,3.795185,3.795185,3.795185,2.209093,...,3.795185,3.795185,3.795185,3.795185,3.795185,3.795185,3.294829,3.294829,3.795185,3.795185
Armenia_2005,2.208708,2.793986,1.015734,3.794525,3.294256,3.294256,3.794525,3.794525,3.794525,2.208708,...,3.794525,3.794525,3.794525,3.794525,3.794525,3.794525,3.294256,3.294256,3.794525,3.794525
Australia_1985,2.208955,2.794297,1.015847,3.794948,3.294623,3.294623,3.794948,3.794948,3.794948,2.208955,...,3.794948,3.794948,3.794948,3.794948,3.794948,3.794948,3.294623,3.294623,3.794948,3.794948


In [30]:
TFIDF_L2.head(10)

term_str,honor,labor,occasion,stage,integral,construction,class,rural,reading,delivered,...,engage,punished,functioning,includes,affect,maximum,f,equally,restrictions,needs
constitution_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0.016776,0.016835,0.016776,0.016776,0.016776,0.016776,0.016776,0.016776,0.016776,0.016776,...,0.008528,0.008542,0.008498,0.008513,0.008498,0.008513,0.008498,0.008498,0.008513,0.008498
Albania_2008,0.016820,0.016885,0.016777,0.016777,0.016777,0.016777,0.016777,0.016777,0.016777,0.016777,...,0.008498,0.008509,0.008498,0.008509,0.008509,0.008498,0.008618,0.008509,0.008498,0.008520
Algeria_2008,0.016821,0.016778,0.016800,0.016778,0.016800,0.016800,0.016778,0.016778,0.016778,0.016800,...,0.008499,0.008531,0.008607,0.008499,0.008499,0.008564,0.008499,0.008499,0.008499,0.008499
Andorra_1993,0.016777,0.016777,0.016777,0.016777,0.016777,0.016777,0.016777,0.016777,0.016777,0.016777,...,0.008498,0.008498,0.008650,0.008498,0.008498,0.008544,0.008589,0.008498,0.008498,0.008513
Angola_2010,0.016773,0.016773,0.016785,0.016773,0.016785,0.016785,0.016773,0.016820,0.016773,0.016785,...,0.008508,0.008502,0.008716,0.008502,0.008508,0.008502,0.008692,0.008502,0.008544,0.008514
Antigua_and_Barbuda_1981,0.016770,0.016770,0.016848,0.016792,0.016770,0.016770,0.016859,0.016770,0.016815,0.016781,...,0.008500,0.008500,0.008495,0.008585,0.008512,0.008500,0.008562,0.008506,0.008574,0.008495
Argentina_1994,0.016819,0.016843,0.016819,0.016771,0.016819,0.016819,0.016771,0.016771,0.016771,0.016771,...,0.008507,0.008519,0.008495,0.008495,0.008519,0.008495,0.008495,0.008495,0.008495,0.008519
Armenia_2005,0.016833,0.016797,0.016779,0.016779,0.016779,0.016797,0.016779,0.016779,0.016779,0.016779,...,0.008517,0.008499,0.008517,0.008508,0.008499,0.008517,0.008508,0.008499,0.008535,0.008517
Australia_1985,0.016774,0.016774,0.016774,0.016796,0.016774,0.016818,0.016818,0.016774,0.016774,0.016796,...,0.008497,0.008497,0.008497,0.008508,0.008519,0.008541,0.008508,0.008519,0.008508,0.008497


# Save

The export lines are intentionally commented out so the derived tables can be inspected before saving.

In [32]:
save_path = f"{output_dir}/{data_prefix}"

BOW.to_csv(f"{save_path}-BOW-{bag}.csv", sep=CSV_DELIM)
DTM.to_csv(f"{save_path}-DTM-{bag}.csv", sep=CSV_DELIM)
DOC.to_csv(f"{save_path}-DOC-{bag}.csv", sep=CSV_DELIM)
TFIDF.to_csv(f"{save_path}-TFIDF-{bag}.csv", sep=CSV_DELIM)
VOCAB.to_csv(f"{save_path}-VOCAB-{bag}.csv", sep=CSV_DELIM)
SIGS.to_csv(f"{save_path}-SIGS-{bag}.csv", sep=CSV_DELIM)
TFIDF_REDUCED.to_csv(f"{save_path}-TFIDF_REDUCED-{bag}.csv", sep=CSV_DELIM)
TFIDF_L2.to_csv(f"{save_path}-TFIDF_L2-{bag}.csv", sep=CSV_DELIM)